# 05t — Transformer

Causal self-attention over the same 8-state window the recurrent models use. The Lorenz flow is
Markov, so there is nothing in that window for attention to find — this is the third and largest
control on how much apparent improvement in this deck is architecture rather than noise.

It carries ~100k parameters against the MLP's 17k. The shared budget is in *iterations*, not
in capacity, so read this as an architecture control and not as a matched comparison.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
for key in DATA:
    print(f"--- {key.upper()} ---")
    for name in (f'05t_transformer_{key}', f'05_lstm_{key}', f'02_mlp_{key}'):
        s = S.get(name)
        if s is None: continue
        print(f"  {name:24s} {s['n_params']:7d} params   horizon {num(s['horizon'],4,0)} "
              f"{rng(s['horizon'],0):>12s}   chaos {num(s['chaos'])}   alive {num(s['alive'])}")

--- ODE ---
  05t_transformer_ode       100163 params   horizon  235    [202–330]   chaos   0.70   alive   0.84
  05_lstm_ode                17859 params   horizon  440    [327–512]   chaos   1.00   alive   1.00
  02_mlp_ode                 17411 params   horizon  336    [242–399]   chaos   1.00   alive   1.00
--- SDE ---
  05t_transformer_sde       100163 params   horizon   49      [32–58]   chaos  -0.54   alive   0.00
  05_lstm_sde                17859 params   horizon   45      [34–48]   chaos   1.03   alive   1.00
  02_mlp_sde                 17411 params   horizon   41      [40–41]   chaos   0.42   alive   0.19
--- SDE015 ---
  02_mlp_sde015              17411 params   horizon  110    [105–119]   chaos   0.94   alive   1.00


## This model

In [3]:
KEY = 'ode'      # any of DATA
s = S['05t_transformer_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('05t_transformer_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

100,163 parameters, history 8, figures show seed 3

ruler             median   range over seeds
  horizon            235   [202–330] over 5 seeds
  spread               —   — over 0 seeds
  climate          20.43   [16.54–31.27] over 5 seeds
  climate_vs_truth   32.93   [26.65–50.39] over 5 seeds
  chaos             0.70   [0.57–0.87] over 5 seeds
  alive             0.84   [0.62–1.00] over 5 seeds
  lobe              0.61   [0.38–0.74] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  7.591e-02
  n=2  1.379e-01
  n=3  1.978e-01
  n=4  2.588e-01
  n=5  3.033e-01
  n=6  3.419e-01


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [4]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50250/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50250/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50250/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 